# EDA courte — HPP Lean (admission → HPPsev)

Objectif : taux de positifs, qualité (NA), **lien features → cible**, redondances.  
Le preprocessing / modèle restent dans `src/train.py`.

L’EDA archivée (Colab, χ² en batterie, SMOTE) n’est **pas** recopiée : voir `docs/COMPARAISON_EDA.md`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.config import FEATURES, NUM_FEATURES, TARGET, PROCESSED_DIR, DEFAULT_RAW_CSV
from src.prepare_data import prepare

processed = PROCESSED_DIR / "admission_hpp.csv"
if processed.exists():
    df = pd.read_csv(processed)
else:
    df = prepare(DEFAULT_RAW_CSV)

print(df.shape)
df.head()

In [ ]:
vc = df[TARGET].value_counts(dropna=False)
rate = df[TARGET].mean()
print(vc)
print(f"Taux {TARGET}+ : {100 * rate:.2f} %")

vc.plot(kind="bar", title=f"Distribution {TARGET}", color=["#4C78A8", "#E45756"])
plt.xticks(rotation=0)
plt.ylabel("Effectif")
plt.show()

In [ ]:
na = df[FEATURES].isna().mean().sort_values(ascending=False)
print(na.head(10).to_string())
na.plot(kind="barh", figsize=(8, 6), title="% NA par feature")
plt.xlabel("% manquant")
plt.tight_layout()
plt.show()

## Dictionnaire (variables du modèle uniquement)

Pas les ~120 colonnes du brut : uniquement ce que `src/config.py` envoie au modèle.

In [ ]:
dico_path = ROOT / "data" / "dico_features_lean.csv"
dico = pd.read_csv(dico_path)
dico

## Lien métier : taux d’HPPsev selon les FdR d’admission

C’est l’analyse que l’archive faisait via χ² / Cramer, racontée en **taux** (plus utile à l’oral). Un FdR rare peut avoir un fort taux et peu d’impact populationnel.

In [ ]:
key_cats = [c for c in ["creta", "hellp", "g_type", "preecl", "ut_cica", "hta_tot", "pma", "cortico"] if c in df.columns]
rows = []
for col in key_cats:
    g = df.groupby(col, dropna=False)[TARGET].agg(["mean", "sum", "count"])
    g.columns = ["taux", "n_hpp", "n"]
    g = g.assign(feature=col).reset_index().rename(columns={col: "valeur"})
    rows.append(g)
rates = pd.concat(rows, ignore_index=True)
rates["taux_pct"] = (100 * rates["taux"]).round(2)
print(rates[["feature", "valeur", "n", "n_hpp", "taux_pct"]].to_string(index=False))

if "creta" in rates["feature"].values:
    ax = rates[rates["feature"] == "creta"].plot(
        x="valeur", y="taux", kind="bar", legend=False, title="Taux HPPsev selon creta (accreta)"
    )
    ax.set_ylabel("Taux")
    plt.show()

## Anti-leakage (à dire à l’oral)

- Features = infos **admission / pré-accouchement** uniquement (`src/config.py`).
- Exclus : transfusion, embolisation, hystérectomie, scores néonataux, etc. (post-événement).
- L’EDA archivée gardait encore `hdd` (hémorragie de la délivrance) et des codes néonataux (`dbp`, `hiv`) : **volontairement absents** du Lean.
- Split **stratifié** + Imputer / Scaler / OHE **fit sur le train seulement** (`Pipeline` dans `train.py`).
- Seuil τ calibré sur **validation**, métriques annoncées sur **test** une seule fois.
- NA : on n’imite pas le dropna global de l’archive ; l’imputation est dans le pipeline.

## Numériques selon la cible

Même idée que les histogrammes d’archive, mais **conditionnés à HPPsev** (médiane plus parlante que la moyenne si outliers).

In [ ]:
print(df.groupby(TARGET)[NUM_FEATURES].median().round(2).T.to_string())

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, col in zip(axes, ["bmi", "age_m", "terme"]):
    if col not in df.columns:
        continue
    df.boxplot(column=col, by=TARGET, ax=ax)
    ax.set_title(col)
    ax.set_xlabel(TARGET)
fig.suptitle("")
plt.tight_layout()
plt.show()

## Redondance AMP / PMA (héritage archive)

L’EDA complète avait déjà viré `Aide_procreation` et `poids_mere` / `Dosecortico`. Lean a encore `AMP` (libellé) et `pma` (0/1) : à connaître pour l’oral, pas à « réparer » en urgence.

In [ ]:
if {"AMP", "pma"}.issubset(df.columns):
    print(pd.crosstab(df["AMP"].fillna("NA"), df["pma"], dropna=False))
    print("\nTaux HPPsev :")
    print(df.groupby(["pma", "AMP"], dropna=False)[TARGET].mean().mul(100).round(2))